# MP2a: Legal-AI Discourse Pipeline (Reddit + Web)

**Research Question:** What recurring themes emerge in public discourse about legal AI tools, and what do they reveal about practitioner sentiment and adoption barriers?

**Data Sources:**
- Reddit API (r/LawFirm, r/lawyers, r/LegalTech, r/artificial, r/ChatGPT)
- Google search scraping (legal-tech blogs, news, academic articles)

**Pipeline:** Collect → Clean → Code Themes → Analyze → Qualitative Netnographic Layer

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Data Collection

Run the collection scripts if data doesn't exist yet. Phase 1A fetches Reddit posts and comments via PRAW. Phase 1B scrapes Google search results for legal-tech articles.

In [ ]:
if not os.path.exists('data/raw_reddit.csv'):
    print('Running Reddit collection...')
    %run 01_collect_reddit.py
else:
    print('Reddit data already exists.')

if not os.path.exists('data/raw_web.csv'):
    print('Running web collection...')
    %run 01_collect_web.py
else:
    print('Web data already exists.')

In [ ]:
df_raw = pd.read_csv('data/raw_combined.csv')
print(f'Total raw rows: {len(df_raw)}')
print(f"\nBy source:")
print(df_raw['source'].value_counts())
print(f"\nReddit subreddit distribution:")
reddit_rows = df_raw[df_raw['source'] == 'reddit']
print(reddit_rows['subreddit'].value_counts())
print(f"\nWeb source types:")
web_rows = df_raw[df_raw['source'] == 'web']
if len(web_rows) > 0:
    print(web_rows['source_type'].value_counts())

## 3. Data Cleaning

Normalize text from both sources: lowercase, remove URLs, strip HTML/markdown artifacts, drop short and duplicate entries.

In [ ]:
if not os.path.exists('data/cleaned_posts.csv'):
    %run 02_clean.py

df_clean = pd.read_csv('data/cleaned_posts.csv')
print(f'Cleaned rows: {len(df_clean)}')
print(f"Reddit: {len(df_clean[df_clean['source'] == 'reddit'])}")
print(f"Web: {len(df_clean[df_clean['source'] == 'web'])}")
print(f"\nSample cleaned text:")
df_clean['clean_text'].head(3).tolist()

## 4. Theme Coding

Two-pass coding: keyword-based assignment first (interpretable, controllable), then TF-IDF + KMeans clustering for posts that don't match any keyword theme.

In [ ]:
if not os.path.exists('data/coded_posts.csv'):
    %run 03_code_themes.py

df_coded = pd.read_csv('data/coded_posts.csv')
print(f'Coded rows: {len(df_coded)}')
print(f"\nTheme distribution:")
theme_counts = df_coded['primary_theme'].value_counts()
print(theme_counts)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
theme_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Posts/Articles')
ax.set_ylabel('Theme')
ax.set_title('Theme Frequency in Legal-AI Discourse')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('data/theme_frequency.png', dpi=150)
plt.show()

In [ ]:
cross_source = df_coded.groupby(['primary_theme', 'source']).size().unstack(fill_value=0)
print('Cross-source validation (themes appearing in both Reddit and Web are stronger signals):')
print(cross_source)

if 'reddit' in cross_source.columns and 'web' in cross_source.columns:
    cross_source.plot(kind='barh', stacked=True, figsize=(10, 6), color=['steelblue', 'coral'])
    plt.xlabel('Count')
    plt.ylabel('Theme')
    plt.title('Theme Frequency by Source (Reddit vs Web)')
    plt.gca().invert_yaxis()
    plt.legend(title='Source')
    plt.tight_layout()
    plt.savefig('data/theme_by_source.png', dpi=150)
    plt.show()

## 5. Analysis & Findings

Frequency-ranked themes with illustrative excerpts and source links.

In [ ]:
if not os.path.exists('data/themes_summary.csv'):
    %run 04_analyze.py

df_summary = pd.read_csv('data/themes_summary.csv')
df_summary[['theme', 'count', 'percentage', 'reddit_count', 'web_count', 'mean_score']]

In [ ]:
for _, row in df_summary.head(5).iterrows():
    print(f"\n{'='*60}")
    print(f"THEME: {row['theme']} ({row['count']} items, {row['percentage']}%)")
    print(f"Reddit: {row['reddit_count']} | Web: {row['web_count']}")
    print(f"-"*60)
    for i in range(1, 4):
        excerpt = row.get(f'excerpt_{i}', '')
        url = row.get(f'excerpt_{i}_url', '')
        if pd.notna(excerpt) and excerpt:
            print(f"\n  Excerpt {i}: \"{excerpt[:200]}...\"")
            if pd.notna(url) and url:
                print(f"  Source: {url}")
    print(f"\n  Memo: {row['memo']}")

## 6. Qualitative Netnographic Layer

Beyond theme frequency, this layer applies Kozinets-informed netnographic methods to capture the cultural and experiential dimensions of legal-AI discourse:

- **Sentiment & Emotion** — VADER polarity + emotion keyword tagging (frustration, enthusiasm, anxiety, skepticism, pragmatism)
- **Speaker Roles** — Who is talking? (practitioner, law student, vendor/builder, tech-adjacent, client/public)
- **Rhetorical Framing** — How do they position claims? (lived experience, fear/warning, hype, measured evaluation, question-seeking, authority citation)
- **Community Norms** — What does the community reward (upvote) vs punish (downvote)?
- **Narrative Extraction** — First-person experience stories with source links
- **Reflexive Memos** — Interpretive summaries per theme combining all qualitative signals

In [ ]:
if not os.path.exists('data/qualitative_coded.csv'):
    %run 05_qualitative.py

df_qual = pd.read_csv('data/qualitative_coded.csv')
print(f'Qualitative-coded rows: {len(df_qual)}')
print(f"\nSentiment distribution:")
print(df_qual['sentiment_label'].value_counts())
print(f"\nDominant emotions:")
print(df_qual['dominant_emotion'].value_counts())
print(f"\nSpeaker roles:")
print(df_qual['speaker_role'].value_counts())
print(f"\nRhetorical frames:")
print(df_qual['rhetorical_frame'].value_counts())

In [ ]:
# Sentiment by theme heatmap
import numpy as np

sentiment_cross = pd.crosstab(df_qual['primary_theme'], df_qual['sentiment_label'], normalize='index')
sentiment_cross = sentiment_cross.reindex(columns=['negative', 'neutral', 'positive'], fill_value=0)

fig, ax = plt.subplots(figsize=(10, 6))
sentiment_cross.plot(kind='barh', stacked=True, ax=ax, 
                     color=['#e74c3c', '#95a5a6', '#2ecc71'])
ax.set_xlabel('Proportion')
ax.set_ylabel('Theme')
ax.set_title('Sentiment Distribution by Theme')
ax.invert_yaxis()
ax.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('data/sentiment_by_theme.png', dpi=150)
plt.show()

In [ ]:
# Speaker roles and dominant emotions side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

role_counts = df_qual['speaker_role'].value_counts()
role_counts.plot(kind='barh', ax=axes[0], color='#3498db')
axes[0].set_title('Speaker Role Distribution')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()

emotion_counts = df_qual[df_qual['dominant_emotion'] != 'none']['dominant_emotion'].value_counts()
emotion_counts.plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Dominant Emotion Distribution')
axes[1].set_xlabel('Count')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('data/roles_emotions.png', dpi=150)
plt.show()

In [ ]:
# Community norms: what does the community reward vs punish?
if os.path.exists('data/community_norms.csv'):
    df_norms = pd.read_csv('data/community_norms.csv')
    print("Community Norms by Theme")
    print("=" * 60)
    for _, row in df_norms.iterrows():
        print(f"\n{row['theme']}:")
        print(f"  Median score: {row['median_score']}")
        print(f"  Rewarded frames: {row['rewarded_frames']}")
        print(f"  Punished frames: {row['punished_frames']}")
        print(f"  Rewarded sentiment avg: {row['rewarded_sentiment_avg']}")
        print(f"  Punished sentiment avg: {row['punished_sentiment_avg']}")
else:
    print("No community norms data found. Run 05_qualitative.py first.")

In [ ]:
# Narrative extraction: first-person experience stories with sources
if os.path.exists('data/narratives.csv'):
    df_narr = pd.read_csv('data/narratives.csv')
    print(f"Extracted {len(df_narr)} first-person narratives across {df_narr['theme'].nunique()} themes\n")
    for theme in df_narr['theme'].unique()[:5]:
        subset = df_narr[df_narr['theme'] == theme]
        print(f"\n{'='*60}")
        print(f"NARRATIVES: {theme} ({len(subset)} stories)")
        print(f"{'='*60}")
        for _, row in subset.head(3).iterrows():
            role_tag = f"[{row['speaker_role']}]" if row['speaker_role'] != 'unidentified' else ""
            print(f"\n  {role_tag} ({row['sentiment']}, {row['emotion']})")
            print(f"  \"{row['excerpt'][:250]}...\"")
            if pd.notna(row['source_url']) and row['source_url']:
                print(f"  Source: {row['source_url']}")
else:
    print("No narratives data found. Run 05_qualitative.py first.")

In [ ]:
# Reflexive memos: interpretive netnographic summaries per theme
if os.path.exists('data/reflexive_memos.csv'):
    df_memos = pd.read_csv('data/reflexive_memos.csv')
    print(f"Generated {len(df_memos)} reflexive memos\n")
    for _, row in df_memos.iterrows():
        print(row['memo'])
        print()
else:
    print("No reflexive memos found. Run 05_qualitative.py first.")

## 7. Discussion

### Key Findings

Interpret the top themes here after running the pipeline on real data. Themes that appear in **both** Reddit and web sources provide stronger evidence — cross-source validation reduces the risk of platform-specific bias.

### Netnographic Insights

The qualitative layer reveals dimensions invisible to pure frequency analysis:
- **Who is speaking** matters: practitioner narratives carry different weight than vendor promotion or student speculation.
- **Emotional tone** varies by theme: some topics (e.g., job displacement) trigger anxiety, while others (efficiency gains) trigger pragmatism.
- **Community norms** surface through upvote patterns: the community rewards lived-experience framing and punishes hype/promotion.
- **First-person narratives** ground abstract themes in concrete practitioner experiences.

### Limitations

- **Reddit bias:** Skews younger, English-speaking, and tech-savvy. Solo practitioners and older attorneys are underrepresented.
- **Web scraping gaps:** Paywalled content (e.g., Law.com premium) is excluded. Some sites block automated scraping.
- **Keyword coding:** Depends on the quality of the keyword dictionary. Nuanced posts that discuss themes without using expected keywords may be miscoded.
- **Role classification:** Regex-based role detection catches explicit self-identification only; many speakers remain unidentified.
- **Temporal snapshot:** Data reflects discourse at the time of collection, not longitudinal trends.

## 8. Future Work

- **Additional sources:** LinkedIn posts, legal-tech conference transcripts, bar association publications
- **Longitudinal tracking:** Re-run collection monthly to track how discourse evolves
- **Interactive explorer:** Build a Dash app for filtering by theme, source, date, emotion, and speaker role
- **LLM-assisted coding:** Use Claude or GPT to generate richer thematic memos from excerpts
- **Network analysis:** Map reply chains to identify opinion leaders and community sub-groups
- **Comparative netnography:** Compare discourse norms across subreddits (e.g., r/LawFirm vs r/LegalTech)